# RongRAG Studio：从文档到可解释答案

这个 Notebook 只负责解释核心链路，实际应用代码位于 `backend/app`。示例使用项目作者自行编写的 `knowledge/project-profile.md`。

In [ ]:
# Cell 1：解析文档并切片
# 输入是文件名和字节内容；输出是带段落、页码、稳定 ID 的 chunks。
# metadata 让答案能够回到原文；切片太大不精准，太小则会丢失上下文。
from pathlib import Path
from backend.app.chunking import split_sections, stable_document_id
from backend.app.parsers import parse_document

path = Path('../knowledge/project-profile.md')
content = path.read_bytes()
document_id = stable_document_id(path.name, content)
sections = parse_document(path.name, content)
chunks = split_sections(document_id, path.name, sections, chunk_size=700, overlap=100)
[(chunk.chunk_id, chunk.metadata()) for chunk in chunks[:2]]

In [ ]:
# Cell 2：文本转向量
# 中文 Embedding 模型把语义映射到固定维度空间，入库和查询必须使用同一模型与归一化方式。
from backend.app.models import get_embedding_model

embedder = get_embedding_model()
embeddings = embedder.encode([chunk.text for chunk in chunks])
assert len(chunks) == len(embeddings)
len(embeddings), len(embeddings[0])

In [ ]:
# Cell 3：写入持久化 ChromaDB
# documents、embeddings、metadatas 和 ids 必须等长且按位置对应。
from backend.app.config import get_settings
from backend.app.store import ChromaStore

settings = get_settings()
store = ChromaStore(settings.chroma_path, settings.collection_name, settings.embedding_model)
store.upsert(chunks, embeddings)
store.count()

In [ ]:
# Cell 4：向量召回是快速粗筛
# 查询向量从全库召回 top-k 候选，同时保留来源 metadata 和相似度分数。
question = '项目如何评估检索效果？'
query_embedding = embedder.encode([question])[0]
retrieved = store.query(query_embedding, limit=10)
[(item.metadata['filename'], item.retrieval_score, item.text[:50]) for item in retrieved]

In [ ]:
# Cell 5：CrossEncoder 精排
# 精排联合阅读问题与候选片段，成本高于向量检索，因此只处理粗筛结果。
# metadata 必须与 chunk 一起排序，否则来源会错位。
from backend.app.models import get_reranker

reranker = get_reranker()
scores = reranker.score(question, [item.text for item in retrieved])
for item, score in zip(retrieved, scores):
    item.rerank_score = score
ranked = sorted(retrieved, key=lambda item: item.rerank_score, reverse=True)[:5]
[(item.metadata['filename'], item.rerank_score) for item in ranked]

In [ ]:
# Cell 6：带来源的生成封装
# Prompt 约束模型只根据证据回答；结构化结果包含答案、来源、模型与各阶段 latency。
# Gemini 响应可能含 thought_signature，因此应用显式读取 candidates[0].content.parts，而非直接访问 response.text。
from backend.app.models import get_generator
from backend.app.service import _build_prompt

generator = get_generator()
answer, raw_response = generator.generate(_build_prompt(question, ranked))
print(answer)
print([item.metadata for item in ranked])